In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import random, re

import torch
import torch.nn as nn

from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

In [3]:
random.seed(33)
np.random.seed(33)
torch.manual_seed(33)

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(33)

In [4]:
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [6]:
OPTIONS = ["A", "B", "C", "D", "E"]

def split_prompt(prompt):
    text = str(prompt).strip()
    prefix, suffix = '', ''

    ci = text.find(': ')
    if ci != -1:
        prefix = text[:ci].strip()
        text = text[ci + 1:].strip()

    n_q = text.count('?')

    if n_q == 1:
        qi = text.index('?')
        suffix = text[qi + 1:].strip()
        text = text[:qi + 1].strip()

    elif n_q == 2:
        first_q = text.index('?')
        prefix = text[:first_q + 1].strip()

        text = text[first_q + 1:].strip()
        second_q = text.index('?')

        suffix = text[second_q + 1:].strip()
        text = text[:second_q + 1].strip()

    return (prefix or None), text, (suffix or None)


def clean(text, is_prompt=True):
    if(is_prompt):
        text = split_prompt(text)[1]
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# def tokenize(text, is_prompt=True):
#     return clean(text, is_prompt).split()


def ap_at_3(actual, predicted):
    for i, p in enumerate(predicted[:3]):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0


def map_at_3(actuals, predictions):
    return float(np.mean([ap_at_3(a, p) for a, p in zip(actuals, predictions)]))


def make_submission(ids, predictions):
    return pd.DataFrame({
        'id': ids,
        'prediction': [' '.join(p[:3]) for p in predictions]
    })

In [7]:
df['q_clean'] = df['prompt'].apply(clean)

for opt in OPTIONS:
    df[f'{opt}_clean'] = df[opt].apply(lambda text: clean(text, is_prompt=False))

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       2000 non-null   int64 
 1   prompt   2000 non-null   object
 2   A        2000 non-null   object
 3   B        2000 non-null   object
 4   C        2000 non-null   object
 5   D        2000 non-null   object
 6   E        2000 non-null   object
 7   answer   2000 non-null   object
 8   q_clean  2000 non-null   object
 9   A_clean  2000 non-null   object
 10  B_clean  2000 non-null   object
 11  C_clean  2000 non-null   object
 12  D_clean  2000 non-null   object
 13  E_clean  2000 non-null   object
dtypes: int64(1), object(13)
memory usage: 218.9+ KB


In [8]:
print("Empty questions:", (df['q_clean'].str.len() == 0).sum())

for opt in OPTIONS:
    empty_count = (df[f'{opt}_clean'].str.len() == 0).sum()
    print(opt, empty_count)

Empty questions: 0
A 0
B 0
C 0
D 0
E 0


In [9]:
unique_qs = df['q_clean'].unique()
(len(df), len(unique_qs), len(set(unique_qs)))

(2000, 252, 252)

In [10]:
NUM_FOLDS = 5

rng = np.random.default_rng(33)
shuffled_qs = unique_qs.copy()
rng.shuffle(shuffled_qs)

fold_qs = np.array_split(shuffled_qs, NUM_FOLDS)

for fold in range(NUM_FOLDS):
    print(f"Fold {fold}: {len(fold_qs[fold])} validation questions")

Fold 0: 51 validation questions
Fold 1: 51 validation questions
Fold 2: 50 validation questions
Fold 3: 50 validation questions
Fold 4: 50 validation questions


In [11]:
def create_pair_dataframe(source_df):
    pair_rows = []

    for row in source_df.itertuples(index=False):
        for opt in OPTIONS:
            pair_rows.append({
                'id': row.id,
                'option': opt,
                'question': row.q_clean,
                'option_text': getattr(row, f'{opt}_clean'),
                'label': int(row.answer == opt)
            })

    return pd.DataFrame(pair_rows)

In [12]:
fold0_val_qs = set(fold_qs[0])
fold0_train_qs = set(unique_qs) - fold0_val_qs

fold0_train_df = df[df['q_clean'].isin(fold0_train_qs)].reset_index(drop=True)
fold0_pairs = create_pair_dataframe(fold0_train_df)

print(fold0_pairs.head(10))

   id option                                           question  \
0   1      A  what is martin heidegger s view on the relatio...   
1   1      B  what is martin heidegger s view on the relatio...   
2   1      C  what is martin heidegger s view on the relatio...   
3   1      D  what is martin heidegger s view on the relatio...   
4   1      E  what is martin heidegger s view on the relatio...   
5   4      A  what is martin heidegger s view on the relatio...   
6   4      B  what is martin heidegger s view on the relatio...   
7   4      C  what is martin heidegger s view on the relatio...   
8   4      D  what is martin heidegger s view on the relatio...   
9   4      E  what is martin heidegger s view on the relatio...   

                                         option_text  label  
0  martin heidegger believes that humans exist wi...      0  
1  martin heidegger believes that humans do not e...      1  
2  martin heidegger does not believe in the exist...      0  
3  martin heid

In [ ]:
MODEL_NAME = 'microsoft/deberta-v3-small'
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_size = min(500, len(fold0_pairs))

sample_lengths = [
    len(tokenizer(row.question, row.option_text)['input_ids'])
    for row in fold0_pairs.sample(sample_size, random_state=33).itertuples(index=False)
]

pd.Series(sample_lengths).describe(percentiles=[0.50, 0.90, 0.95, 0.99])

In [14]:
def encode_groups(pair_df, tokenizer, max_len, has_labels=True):
    samples = []

    for question_id, group in pair_df.groupby('id', sort=False):
        group = group.sort_values('option')

        encoded = tokenizer(
            group['question'].tolist(),
            group['option_text'].tolist(),
            truncation=True,
            max_length=max_len,
            padding='max_length',
            return_tensors='pt'
        )

        sample = {
            'input_ids': encoded['input_ids'],
            'attention_mask': encoded['attention_mask'],
            'id': question_id
        }

        if has_labels:
            label_idx = group['label'].tolist().index(1)
            sample['label'] = torch.tensor(label_idx, dtype=torch.long)

        samples.append(sample)

    return samples

In [15]:
BATCH_SIZE = 8

In [16]:
class DebertaMCQ(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name, dtype='float32')
        hidden_dim = self.encoder.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, input_ids, attention_mask):
        batch_size, num_options, seq_len = input_ids.shape

        input_ids = input_ids.view(batch_size * num_options, seq_len)
        attention_mask = attention_mask.view(batch_size * num_options, seq_len)

        encoder_output = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = encoder_output.last_hidden_state[:, 0, :]

        logits = self.classifier(cls_output).squeeze(-1)
        logits = logits.view(batch_size, num_options)

        return logits

In [ ]:
sanity_model = DebertaMCQ(MODEL_NAME).to(device)

fold0_train_samples = encode_groups(fold0_pairs, tokenizer, MAX_LEN, has_labels=True)
fold0_train_loader = DataLoader(fold0_train_samples, batch_size=BATCH_SIZE, shuffle=True)

batch = next(iter(fold0_train_loader))

input_ids = batch['input_ids'].to(device)
attention_mask = batch['attention_mask'].to(device)

with torch.no_grad():
    logits = sanity_model(input_ids, attention_mask)

print("Logits shape:", logits.shape)
print("Example logits (5 options of one question):", logits[0])

In [18]:
LEARNING_RATE = 2e-5

criterion = nn.CrossEntropyLoss()

In [19]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, total_samples = 0.0, 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        batch_size = input_ids.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples

In [20]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    predictions, actual_answers = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()

        batch_size = input_ids.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        labels_cpu = labels.cpu().numpy()

        for probability_row, label in zip(probabilities, labels_cpu):
            ranked_options = [OPTIONS[idx] for idx in np.argsort(-probability_row)]
            predictions.append(ranked_options[:3])
            actual_answers.append(OPTIONS[label])

    return {
        'loss': total_loss / total_samples,
        'map3': map_at_3(actual_answers, predictions),
        'predictions': predictions,
        'actual_answers': actual_answers
    }

In [21]:
EPOCHS = 10

def run_fold(fold, fold_qs):
    val_qs = set(fold_qs[fold])
    train_qs = set(unique_qs) - val_qs

    train_df = df[df['q_clean'].isin(train_qs)].reset_index(drop=True)
    val_df = df[df['q_clean'].isin(val_qs)].reset_index(drop=True)

    train_pairs = create_pair_dataframe(train_df)
    val_pairs = create_pair_dataframe(val_df)

    train_samples = encode_groups(train_pairs, tokenizer, MAX_LEN, has_labels=True)
    val_samples = encode_groups(val_pairs, tokenizer, MAX_LEN, has_labels=True)

    train_loader = DataLoader(train_samples, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_samples, batch_size=BATCH_SIZE*2, shuffle=False)

    model = DebertaMCQ(MODEL_NAME).to(device)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    best_map3 = -1.0
    best_epoch = 0

    history = []
    checkpoint_path = f'best_deberta_fold{fold}.pt'

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
        validation_result = evaluate(model, val_loader, criterion, device)

        val_loss = validation_result['loss']
        val_map3 = validation_result['map3']

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_map3': val_map3
        })

        print(f"Fold {fold}: Epoch {epoch} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val MAP@3: {val_map3:.4f}")

        if val_map3 > best_map3:
            best_map3 = val_map3
            best_epoch = epoch

            torch.save(model.state_dict(), checkpoint_path)
            print(f"Fold {fold}, Best model saved: MAP@3={best_map3:.4f}")

    print(f"\nFold {fold} training complete")
    print(f"Fold {fold}, Best epoch: {best_epoch}")
    print(f"Fold {fold}, Best validation MAP@3: {best_map3}")

    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    model.eval()

    best_validation_result = evaluate(model, val_loader, criterion, device)

    oof_ids = val_pairs['id'].unique().tolist()

    return {
        'fold': fold,
        'history': pd.DataFrame(history),
        'best_map3': best_map3,
        'model': model,
        'oof_ids': oof_ids,
        'oof_predictions': best_validation_result['predictions'],
        'oof_actuals': best_validation_result['actual_answers']
    }

In [ ]:
fold_results = []

for fold in range(NUM_FOLDS):
    fold_results.append(run_fold(fold, fold_qs))

Fold 0: Epoch 1 | Train loss: 1.5361 | Val loss: 1.5076 | Val MAP@3: 0.4849  
Fold 0, Best model saved: MAP@3=0.4849  
Fold 0: Epoch 2 | Train loss: 0.6634 | Val loss: 1.4620 | Val MAP@3: 0.5852  
Fold 0, Best model saved: MAP@3=0.5852  
Fold 0: Epoch 3 | Train loss: 0.2734 | Val loss: 1.6190 | Val MAP@3: 0.6182  
Fold 0, Best model saved: MAP@3=0.6182  
Fold 0: Epoch 4 | Train loss: 0.1152 | Val loss: 1.5904 | Val MAP@3: 0.5745  
Fold 0: Epoch 5 | Train loss: 0.0540 | Val loss: 1.6945 | Val MAP@3: 0.6744  
Fold 0, Best model saved: MAP@3=0.6744  
Fold 0: Epoch 6 | Train loss: 0.0287 | Val loss: 1.8863 | Val MAP@3: 0.6329  
Fold 0: Epoch 7 | Train loss: 0.0166 | Val loss: 1.9240 | Val MAP@3: 0.6318  
Fold 0: Epoch 8 | Train loss: 0.0059 | Val loss: 1.8723 | Val MAP@3: 0.6145  
Fold 0: Epoch 9 | Train loss: 0.0036 | Val loss: 1.9338 | Val MAP@3: 0.6509  
Fold 0: Epoch 10 | Train loss: 0.0042 | Val loss: 1.8855 | Val MAP@3: 0.6399  

Fold 0 training complete  
Fold 0, Best epoch: 5  
Fold 0, Best validation MAP@3: 0.6743759177679882  

Fold 1: Epoch 1 | Train loss: 1.4873 | Val loss: 1.4104 | Val MAP@3: 0.5829  
Fold 1, Best model saved: MAP@3=0.5829  
Fold 1: Epoch 2 | Train loss: 0.7344 | Val loss: 1.4969 | Val MAP@3: 0.6127  
Fold 1, Best model saved: MAP@3=0.6127  
Fold 1: Epoch 3 | Train loss: 0.3223 | Val loss: 1.7320 | Val MAP@3: 0.5768  
Fold 1: Epoch 4 | Train loss: 0.1456 | Val loss: 2.2495 | Val MAP@3: 0.6132  
Fold 1, Best model saved: MAP@3=0.6132  
Fold 1: Epoch 5 | Train loss: 0.0681 | Val loss: 2.2248 | Val MAP@3: 0.5870  
Fold 1: Epoch 6 | Train loss: 0.0312 | Val loss: 3.2219 | Val MAP@3: 0.5458  
Fold 1: Epoch 7 | Train loss: 0.0149 | Val loss: 2.6596 | Val MAP@3: 0.5940  
Fold 1: Epoch 8 | Train loss: 0.0071 | Val loss: 2.9289 | Val MAP@3: 0.5760  
Fold 1: Epoch 9 | Train loss: 0.0092 | Val loss: 3.0900 | Val MAP@3: 0.5641  
Fold 1: Epoch 10 | Train loss: 0.0083 | Val loss: 2.9718 | Val MAP@3: 0.5694  

Fold 1 training complete  
Fold 1, Best epoch: 4  
Fold 1, Best validation MAP@3: 0.6131535947712418  

Fold 2: Epoch 1 | Train loss: 1.4896 | Val loss: 1.3599 | Val MAP@3: 0.5620  
Fold 2, Best model saved: MAP@3=0.5620  
Fold 2: Epoch 2 | Train loss: 0.6677 | Val loss: 1.7303 | Val MAP@3: 0.5847  
Fold 2, Best model saved: MAP@3=0.5847  
Fold 2: Epoch 3 | Train loss: 0.2810 | Val loss: 2.0203 | Val MAP@3: 0.6690  
Fold 2, Best model saved: MAP@3=0.6690  
Fold 2: Epoch 4 | Train loss: 0.1439 | Val loss: 2.1328 | Val MAP@3: 0.5859  
Fold 2: Epoch 5 | Train loss: 0.0618 | Val loss: 2.4293 | Val MAP@3: 0.5965  
Fold 2: Epoch 6 | Train loss: 0.0400 | Val loss: 2.5236 | Val MAP@3: 0.5808  
Fold 2: Epoch 7 | Train loss: 0.0092 | Val loss: 2.7600 | Val MAP@3: 0.5965  
Fold 2: Epoch 8 | Train loss: 0.0072 | Val loss: 2.8764 | Val MAP@3: 0.5969  
Fold 2: Epoch 9 | Train loss: 0.0133 | Val loss: 2.9629 | Val MAP@3: 0.6110  
Fold 2: Epoch 10 | Train loss: 0.0114 | Val loss: 2.9319 | Val MAP@3: 0.6169  

Fold 2 training complete  
Fold 2, Best epoch: 3  
Fold 2, Best validation MAP@3: 0.6690196078431373  

Fold 3: Epoch 1 | Train loss: 1.5223 | Val loss: 1.4933 | Val MAP@3: 0.5218  
Fold 3, Best model saved: MAP@3=0.5218  
Fold 3: Epoch 2 | Train loss: 0.7353 | Val loss: 2.0461 | Val MAP@3: 0.5854  
Fold 3, Best model saved: MAP@3=0.5854  
Fold 3: Epoch 3 | Train loss: 0.2855 | Val loss: 2.1388 | Val MAP@3: 0.5831  
Fold 3: Epoch 4 | Train loss: 0.1140 | Val loss: 2.5628 | Val MAP@3: 0.5474  
Fold 3: Epoch 5 | Train loss: 0.0724 | Val loss: 2.9737 | Val MAP@3: 0.5859  
Fold 3, Best model saved: MAP@3=0.5859  
Fold 3: Epoch 6 | Train loss: 0.0338 | Val loss: 3.6061 | Val MAP@3: 0.5543  
Fold 3: Epoch 7 | Train loss: 0.0192 | Val loss: 3.0598 | Val MAP@3: 0.5135  
Fold 3: Epoch 8 | Train loss: 0.0146 | Val loss: 3.5411 | Val MAP@3: 0.5418  
Fold 3: Epoch 9 | Train loss: 0.0089 | Val loss: 3.2942 | Val MAP@3: 0.5292  
Fold 3: Epoch 10 | Train loss: 0.0061 | Val loss: 3.5116 | Val MAP@3: 0.5232  

Fold 3 training complete  
Fold 3, Best epoch: 5  
Fold 3, Best validation MAP@3: 0.585886722376973  

Fold 4: Epoch 1 | Train loss: 1.5344 | Val loss: 1.2781 | Val MAP@3: 0.5927  
Fold 4, Best model saved: MAP@3=0.5927  
Fold 4: Epoch 2 | Train loss: 0.7029 | Val loss: 1.1451 | Val MAP@3: 0.6911  
Fold 4, Best model saved: MAP@3=0.6911  
Fold 4: Epoch 3 | Train loss: 0.2928 | Val loss: 1.3484 | Val MAP@3: 0.6907  
Fold 4: Epoch 4 | Train loss: 0.1173 | Val loss: 1.8320 | Val MAP@3: 0.6676  
Fold 4: Epoch 5 | Train loss: 0.0399 | Val loss: 2.1567 | Val MAP@3: 0.6737  
Fold 4: Epoch 6 | Train loss: 0.0289 | Val loss: 2.4115 | Val MAP@3: 0.6191  
Fold 4: Epoch 7 | Train loss: 0.0157 | Val loss: 2.5396 | Val MAP@3: 0.6351  
Fold 4: Epoch 8 | Train loss: 0.0103 | Val loss: 2.5188 | Val MAP@3: 0.6257  
Fold 4: Epoch 9 | Train loss: 0.0064 | Val loss: 2.6697 | Val MAP@3: 0.6379  
Fold 4: Epoch 10 | Train loss: 0.0090 | Val loss: 2.7386 | Val MAP@3: 0.6540  

Fold 4 training complete  
Fold 4, Best epoch: 2  
Fold 4, Best validation MAP@3: 0.6911487758945386  

In [23]:
oof_predictions, oof_actuals = [], []

for result in fold_results:
    oof_predictions.extend(result['oof_predictions'])
    oof_actuals.extend(result['oof_actuals'])
    print(f"Fold {result['fold']} best validation MAP@3: {result['best_map3']:.4f}")

overall_map3 = map_at_3(oof_actuals, oof_predictions)
print("Overall OOF MAP@3:", overall_map3)

Fold 0 best validation MAP@3: 0.6744
Fold 1 best validation MAP@3: 0.6132
Fold 2 best validation MAP@3: 0.6690
Fold 3 best validation MAP@3: 0.5859
Fold 4 best validation MAP@3: 0.6911
Overall OOF MAP@3: 0.6478333333333334


In [24]:
history_df = pd.concat(
    [result['history'].assign(fold=result['fold']) for result in fold_results],
    ignore_index=True
)
history_df

,epoch,train_loss,val_loss,val_map3,fold
0,1,1.536077,1.507572,0.484949,0
1,2,0.663393,1.462048,0.585169,0
2,3,0.273425,1.619031,0.618209,0
3,4,0.115232,1.590364,0.574523,0
4,5,0.054037,1.694534,0.674376,0
5,6,0.028654,1.886295,0.632893,0
6,7,0.016625,1.923986,0.631791,0
7,8,0.005892,1.872320,0.614537,0
8,9,0.003608,1.933769,0.650881,0
9,10,0.004231,1.885494,0.639868,0


In [25]:
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
test_df['q_clean'] = test_df['prompt'].apply(clean)

for opt in OPTIONS:
    test_df[f'{opt}_clean'] = test_df[opt].apply(lambda text: clean(text, is_prompt=False))

print(test_df.shape)
test_df.head()

(500, 13)


,id,prompt,A,B,C,D,E,q_clean,A_clean,B_clean,C_clean,D_clean,E_clean
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...",what is the relationship between the hamiltoni...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi...",what is the estimated redshift of ceers 93316 ...,approximately z 6 0 corresponding to 1 billion...,approximately z 16 7 corresponding to 235 8 mi...,approximately z 3 0 corresponding to 5 billion...,approximately z 10 0 corresponding to 13 billi...,approximately z 13 0 corresponding to 30 billi...
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...,what is the reason for the sun appearing sligh...,the sun appears yellowish due to a reflection ...,the longer wavelengths of light such as red an...,the sun appears yellowish due to the scatterin...,the sun emits a yellow light due to its own sp...,the atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,what is the significance of the redshift dista...,observations of the redshift distance relation...,observations of the redshift distance relation...,observations of the redshift distance relation...,observations of the redshift distance relation...,observations of the redshift distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,what is the landau lifshitz gilbert equation u...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...


In [26]:
def create_test_pair_dataframe(source_df):
    pair_rows = []

    for row in source_df.itertuples(index=False):
        for opt in OPTIONS:
            pair_rows.append({
                'id': int(row.id),
                'option': opt,
                'question': row.q_clean,
                'option_text': getattr(row, f'{opt}_clean')
            })

    return pd.DataFrame(pair_rows)

In [27]:
test_pairs = create_test_pair_dataframe(test_df)

(len(test_df), len(test_pairs))

(500, 2500)

In [28]:
@torch.no_grad()
def predict_test(model, loader, device):
    model.eval()

    ids_list, probabilities_list = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        logits = model(input_ids, attention_mask)
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()

        ids_cpu = batch['id'].cpu().numpy()

        ids_list.extend(ids_cpu.tolist())
        probabilities_list.extend(probabilities)

    return ids_list, probabilities_list

In [29]:
fold_probabilities = []

for result in fold_results:
    test_samples = encode_groups(test_pairs, tokenizer, MAX_LEN, has_labels=False)
    test_loader = DataLoader(test_samples, batch_size=BATCH_SIZE*2, shuffle=False)

    test_ids, test_probs = predict_test(result['model'], test_loader, device)

    fold_probabilities.append(np.array(test_probs))

ensemble_probabilities = np.mean(fold_probabilities, axis=0)

In [30]:
test_predictions = [
    [OPTIONS[idx] for idx in np.argsort(-probability_row)][:3]
    for probability_row in ensemble_probabilities
]

submission = make_submission(test_ids, test_predictions)
submission.to_csv('submission.csv', index=False)

submission.head(5)

,id,prediction
0,1,A B C
1,2,B D A
2,3,B E D
3,4,E B C
4,5,C A E
